
# Roxy notebook example: AAIndex-based descriptors

This notebook is a **reference implementation example** for the **AAIndex descriptor family** in Roxy.

The implementation uses an **embedded AAIndex class**, generated from a CSV table, so descriptor extraction does **not depend on external runtime files**.

## Covered outputs

This notebook implements:

- an embedded AAIndex data container
- access to AAIndex codes and residue scales
- sequence-level summaries for selected AAIndex encoders
- mean, std, min, max, and median
- optional N-terminal and C-terminal means
- validation and sanity checks
- a class-style implementation for later migration into Roxy

The main goal is to provide a **clean teaching implementation** that can later become the real `aaindex.py` module in Roxy.


In [1]:
import pandas as pd
import numpy as np

from roxy_aaindex_embedded import AAIndexEmbedded


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "aaidx_1",
            "aaidx_2",
            "aaidx_3",
            "aaidx_4",
            "aaidx_5",
            "aaidx_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,aaidx_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,aaidx_2,GGGGGGGGGGGGGGG,B
2,aaidx_3,KRRKRRKRRKRRDDDDEE,A
3,aaidx_4,ACDEFGHIKLMNPQRSTVWY,B
4,aaidx_5,PPPPGSSSSSTTTTNNQQQ,A
5,aaidx_6,MSTNPKPQRITLKDGNKVELV,B


## Inspect the embedded AAIndex resource

In [3]:

available_codes = AAIndexEmbedded.list_indices()
len(available_codes), available_codes[:15]


(566,
 ['ANDN920101',
  'ARGP820101',
  'ARGP820102',
  'ARGP820103',
  'AURR980101',
  'AURR980102',
  'AURR980103',
  'AURR980104',
  'AURR980105',
  'AURR980106',
  'AURR980107',
  'AURR980108',
  'AURR980109',
  'AURR980110',
  'AURR980111'])

In [4]:

example_code = available_codes[0]
example_code, AAIndexEmbedded.get_scale(example_code)


('ANDN920101',
 {'A': 4.35,
  'L': 4.17,
  'R': 4.38,
  'K': 4.36,
  'N': 4.75,
  'M': 4.52,
  'D': 4.76,
  'F': 4.66,
  'C': 4.65,
  'P': 4.44,
  'Q': 4.37,
  'S': 4.5,
  'E': 4.29,
  'T': 4.35,
  'G': 3.97,
  'W': 4.7,
  'H': 4.63,
  'Y': 4.6,
  'I': 3.95,
  'V': 3.95})

## Select a small set of AAIndex encoders for the demo

In [5]:

selected_codes = [
    "ANDN920101",
    "ARGP820101",
    "KYTJ820101",
    "FAUJ880104",
]

selected_codes


['ANDN920101', 'ARGP820101', 'KYTJ820101', 'FAUJ880104']

## Core descriptor function

In [6]:

def aaindex_descriptors(seq: str, codes, include_terminal=True, terminal_window=10) -> dict:
    seq = AAIndexEmbedded.clean_sequence(seq)

    out = {
        "aaidx_length": len(seq),
        "aaidx_valid_residue_count": len(seq),
    }

    for code in codes:
        summary = AAIndexEmbedded.sequence_summary(seq, code)
        out[f"aaidx_{code}_mean"] = summary["mean"]
        out[f"aaidx_{code}_std"] = summary["std"]
        out[f"aaidx_{code}_min"] = summary["min"]
        out[f"aaidx_{code}_max"] = summary["max"]
        out[f"aaidx_{code}_median"] = summary["median"]

        if include_terminal:
            terminal = AAIndexEmbedded.terminal_summary(seq, code, window=terminal_window)
            out[f"aaidx_{code}_nterm_mean_w{terminal_window}"] = terminal["nterm_mean"]
            out[f"aaidx_{code}_cterm_mean_w{terminal_window}"] = terminal["cterm_mean"]

    return out


## Functional usage on one sequence

In [7]:

example = aaindex_descriptors(df_demo.loc[0, "sequence"], selected_codes, include_terminal=True, terminal_window=10)
list(example.items())[:16]


[('aaidx_length', 24),
 ('aaidx_valid_residue_count', 24),
 ('aaidx_ANDN920101_mean', 4.374583333333333),
 ('aaidx_ANDN920101_std', 0.2413672714396511),
 ('aaidx_ANDN920101_min', 3.95),
 ('aaidx_ANDN920101_max', 4.7),
 ('aaidx_ANDN920101_median', 4.38),
 ('aaidx_ANDN920101_nterm_mean_w10', 4.333),
 ('aaidx_ANDN920101_cterm_mean_w10', 4.367),
 ('aaidx_ARGP820101_mean', 1.1300000000000001),
 ('aaidx_ARGP820101_std', 0.812742271571991),
 ('aaidx_ARGP820101_min', 0.05),
 ('aaidx_ARGP820101_max', 2.65),
 ('aaidx_ARGP820101_median', 1.25),
 ('aaidx_ARGP820101_nterm_mean_w10', 1.3699999999999999),
 ('aaidx_ARGP820101_cterm_mean_w10', 0.78)]

## Apply AAIndex descriptors to the full dataset

In [8]:

df_aaidx = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(
            lambda x: aaindex_descriptors(
                x,
                selected_codes,
                include_terminal=True,
                terminal_window=10,
            )
        ).apply(pd.Series),
    ],
    axis=1,
)

df_aaidx.head()


,sequence_id,sequence,label,aaidx_length,aaidx_valid_residue_count,aaidx_ANDN920101_mean,aaidx_ANDN920101_std,aaidx_ANDN920101_min,aaidx_ANDN920101_max,aaidx_ANDN920101_median,...,aaidx_KYTJ820101_median,aaidx_KYTJ820101_nterm_mean_w10,aaidx_KYTJ820101_cterm_mean_w10,aaidx_FAUJ880104_mean,aaidx_FAUJ880104_std,aaidx_FAUJ880104_min,aaidx_FAUJ880104_max,aaidx_FAUJ880104_median,aaidx_FAUJ880104_nterm_mean_w10,aaidx_FAUJ880104_cterm_mean_w10
0,aaidx_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,4.374583,0.241367,3.95,4.70,4.38,...,0.70,1.47,-0.80,5.017500,1.540163,2.06,7.82,4.620,5.250,4.979
1,aaidx_2,GGGGGGGGGGGGGGG,B,15.0,15.0,3.970000,0.000000,3.97,3.97,3.97,...,-0.40,-0.40,-0.40,2.060000,0.000000,2.06,2.06,2.060,2.060,2.060
2,aaidx_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,4.450000,0.167862,4.29,4.76,4.38,...,-3.90,-4.26,-3.84,6.723333,1.215685,4.74,7.82,6.890,7.448,6.125
3,aaidx_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,4.417500,0.249777,3.95,4.76,4.41,...,-0.85,0.09,-1.07,5.013500,1.416888,2.06,7.82,4.735,4.669,5.358
4,aaidx_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,4.433684,0.158520,3.97,4.75,4.44,...,-0.80,-1.08,-2.11,4.330526,0.912437,2.06,6.11,4.110,3.835,4.790


## Inspect descriptor columns

In [9]:

aaidx_cols = [c for c in df_aaidx.columns if c.startswith("aaidx_") and c not in {"aaidx_length", "aaidx_valid_residue_count"}]
len(aaidx_cols), aaidx_cols[:12]


(28,
 ['aaidx_ANDN920101_mean',
  'aaidx_ANDN920101_std',
  'aaidx_ANDN920101_min',
  'aaidx_ANDN920101_max',
  'aaidx_ANDN920101_median',
  'aaidx_ANDN920101_nterm_mean_w10',
  'aaidx_ANDN920101_cterm_mean_w10',
  'aaidx_ARGP820101_mean',
  'aaidx_ARGP820101_std',
  'aaidx_ARGP820101_min',
  'aaidx_ARGP820101_max',
  'aaidx_ARGP820101_median'])

In [10]:

df_aaidx[
    [
        "sequence_id",
        "aaidx_ANDN920101_mean",
        "aaidx_ARGP820101_mean",
        "aaidx_KYTJ820101_mean",
        "aaidx_FAUJ880104_mean",
        "aaidx_ANDN920101_nterm_mean_w10",
        "aaidx_ANDN920101_cterm_mean_w10",
    ]
]


,sequence_id,aaidx_ANDN920101_mean,aaidx_ARGP820101_mean,aaidx_KYTJ820101_mean,aaidx_FAUJ880104_mean,aaidx_ANDN920101_nterm_mean_w10,aaidx_ANDN920101_cterm_mean_w10
0,aaidx_1,4.374583,1.130000,0.637500,5.017500,4.333,4.367
1,aaidx_2,3.970000,0.070000,-0.400000,2.060000,3.970,3.970
2,aaidx_3,4.450000,0.676667,-4.033333,6.723333,4.372,4.512
3,aaidx_4,4.417500,0.997500,-0.490000,5.013500,4.379,4.456
4,aaidx_5,4.433684,0.444211,-1.636842,4.330526,4.423,4.451
5,aaidx_6,4.340000,0.872381,-0.814286,5.060952,4.406,4.273


## Dataset-level summary

In [11]:

aaidx_summary = (
    df_aaidx[aaidx_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

aaidx_summary.head(15)


,descriptor,mean_value
0,aaidx_FAUJ880104_max,6.575000
1,aaidx_FAUJ880104_nterm_mean_w10,4.760000
2,aaidx_FAUJ880104_cterm_mean_w10,4.705167
3,aaidx_FAUJ880104_mean,4.700969
4,aaidx_ANDN920101_max,4.616667
5,aaidx_FAUJ880104_median,4.525833
6,aaidx_ANDN920101_cterm_mean_w10,4.338167
7,aaidx_ANDN920101_mean,4.330961
8,aaidx_ANDN920101_median,4.323333
9,aaidx_ANDN920101_nterm_mean_w10,4.313833


## Sanity checks

In [12]:

assert AAIndexEmbedded.has_index("ANDN920101")
assert AAIndexEmbedded.has_index("ARGP820101")
assert "aaidx_ANDN920101_mean" in df_aaidx.columns
assert "aaidx_ARGP820101_std" in df_aaidx.columns
assert "aaidx_KYTJ820101_nterm_mean_w10" in df_aaidx.columns
assert df_aaidx["aaidx_length"].min() > 0

print(f"Number of embedded AAIndex codes: {len(AAIndexEmbedded.list_indices())}")
print(f"Number of AAIndex descriptor columns in demo table: {len(aaidx_cols)}")
print("AAIndex descriptor checks passed.")


Number of embedded AAIndex codes: 566
Number of AAIndex descriptor columns in demo table: 28
AAIndex descriptor checks passed.


## Class-style implementation closer to the real package

In [13]:

class AAIndexDescriptors:
    """Example class-style AAIndex implementation for later migration into Roxy."""

    def __init__(self, codes, include_terminal=True, terminal_window=10):
        self.codes = list(codes)
        self.include_terminal = include_terminal
        self.terminal_window = terminal_window

    def transform_sequence(self, seq: str) -> dict:
        return aaindex_descriptors(
            seq,
            self.codes,
            include_terminal=self.include_terminal,
            terminal_window=self.terminal_window,
        )

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


aaidx_transformer = AAIndexDescriptors(
    codes=selected_codes,
    include_terminal=True,
    terminal_window=10,
)

aaidx_matrix = aaidx_transformer.transform(df_demo["sequence"].tolist())
aaidx_matrix.head()


,aaidx_length,aaidx_valid_residue_count,aaidx_ANDN920101_mean,aaidx_ANDN920101_std,aaidx_ANDN920101_min,aaidx_ANDN920101_max,aaidx_ANDN920101_median,aaidx_ANDN920101_nterm_mean_w10,aaidx_ANDN920101_cterm_mean_w10,aaidx_ARGP820101_mean,...,aaidx_KYTJ820101_median,aaidx_KYTJ820101_nterm_mean_w10,aaidx_KYTJ820101_cterm_mean_w10,aaidx_FAUJ880104_mean,aaidx_FAUJ880104_std,aaidx_FAUJ880104_min,aaidx_FAUJ880104_max,aaidx_FAUJ880104_median,aaidx_FAUJ880104_nterm_mean_w10,aaidx_FAUJ880104_cterm_mean_w10
0,24,24,4.374583,0.241367,3.95,4.70,4.38,4.333,4.367,1.130000,...,0.70,1.47,-0.80,5.017500,1.540163,2.06,7.82,4.620,5.250,4.979
1,15,15,3.970000,0.000000,3.97,3.97,3.97,3.970,3.970,0.070000,...,-0.40,-0.40,-0.40,2.060000,0.000000,2.06,2.06,2.060,2.060,2.060
2,18,18,4.450000,0.167862,4.29,4.76,4.38,4.372,4.512,0.676667,...,-3.90,-4.26,-3.84,6.723333,1.215685,4.74,7.82,6.890,7.448,6.125
3,20,20,4.417500,0.249777,3.95,4.76,4.41,4.379,4.456,0.997500,...,-0.85,0.09,-1.07,5.013500,1.416888,2.06,7.82,4.735,4.669,5.358
4,19,19,4.433684,0.158520,3.97,4.75,4.44,4.423,4.451,0.444211,...,-0.80,-1.08,-2.11,4.330526,0.912437,2.06,6.11,4.110,3.835,4.790


## Merge transformer output back to the dataset

In [14]:

df_aaidx_class = pd.concat([df_demo, aaidx_matrix], axis=1)
df_aaidx_class.head()


,sequence_id,sequence,label,aaidx_length,aaidx_valid_residue_count,aaidx_ANDN920101_mean,aaidx_ANDN920101_std,aaidx_ANDN920101_min,aaidx_ANDN920101_max,aaidx_ANDN920101_median,...,aaidx_KYTJ820101_median,aaidx_KYTJ820101_nterm_mean_w10,aaidx_KYTJ820101_cterm_mean_w10,aaidx_FAUJ880104_mean,aaidx_FAUJ880104_std,aaidx_FAUJ880104_min,aaidx_FAUJ880104_max,aaidx_FAUJ880104_median,aaidx_FAUJ880104_nterm_mean_w10,aaidx_FAUJ880104_cterm_mean_w10
0,aaidx_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,4.374583,0.241367,3.95,4.70,4.38,...,0.70,1.47,-0.80,5.017500,1.540163,2.06,7.82,4.620,5.250,4.979
1,aaidx_2,GGGGGGGGGGGGGGG,B,15,15,3.970000,0.000000,3.97,3.97,3.97,...,-0.40,-0.40,-0.40,2.060000,0.000000,2.06,2.06,2.060,2.060,2.060
2,aaidx_3,KRRKRRKRRKRRDDDDEE,A,18,18,4.450000,0.167862,4.29,4.76,4.38,...,-3.90,-4.26,-3.84,6.723333,1.215685,4.74,7.82,6.890,7.448,6.125
3,aaidx_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,4.417500,0.249777,3.95,4.76,4.41,...,-0.85,0.09,-1.07,5.013500,1.416888,2.06,7.82,4.735,4.669,5.358
4,aaidx_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,4.433684,0.158520,3.97,4.75,4.44,...,-0.80,-1.08,-2.11,4.330526,0.912437,2.06,6.11,4.110,3.835,4.790



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move the embedded class into `roxy/sequence/aaindex.py`
- keep the embedded AAIndex data inside the module so runtime file dependencies are avoided
- expose helper methods:
  - `list_indices()`
  - `get_scale(code)`
  - `sequence_summary(seq, code)`
- expose a descriptor class such as `AAIndexDescriptors`
- support configurable:
  - selected codes
  - statistics to compute
  - terminal windows
- add tests for:
  - invalid AAIndex codes
  - empty sequences
  - lower-case input
  - sequences containing invalid characters
  - codes with constant or near-constant values


## Optional export

In [ ]:
# df_aaidx.to_csv("demo_aaindex_descriptors.csv", index=False)
